In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_iquam_track_check,
    do_multiple_sequential_check,
    do_spike_check,
)

In [ ]:
from data import get_sequential_data

# How to use quality control checks with sequential reports from ships

The ``marine_qc`` toolbox provides some quality control (QC) checks that work on sequential marine ship reports. Creating some test dataset we can show how to use them on "real" data. There are two checks that are shown here:

* a spike check that tests that a sequence of values has unlikely “spikes” in it.
* a track check that tests that the locations of a series of reports form a plausible ship track

Finally, we run all these QC checks within a single function and combine the results of each QC check into a single QC flag.

For more information of all available QC checks on sequential ship reports see [the overview of QC functions for sequential ship reports](https://marine-qc.readthedocs.io/en/stable/overview_seq.html).

The test dataset we provide is a ``pandas.DataFrame`` including sequential marine reports representing the track of one ship. The QC functions return a QC flag for each individual report. The flags are:

* `0`: the check has passed
* `1`: the check has failed
* `2`: the check is untestable

In [ ]:
data = get_sequential_data()
data

The ship data include four different parameters (`lat`, `lon`, `date`, and `sst`).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))

axs[0].plot(data["lon"], data["lat"], marker="o", linestyle="-")
axs[0].set_title("Ship – latitude/longitude track")
axs[0].set_xlabel("Longitude (°)")
axs[0].set_ylabel("Latitude (°)")
axs[0].grid(True)

axs[1].plot(data["sst"], marker="o", linestyle="-")
axs[1].set_title("Ship – sea surface temperature")
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Temperature (°C)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

Firstly, a spike check is performed. Beside `lat`, `lon`, `date` and `value`, this check needs some extra parameters:

* `max_gradient_space`: Maximum allowed spatial gradient (`0.5`: "0.5 K per kilometer")
* `max_gradient_time`: Maximum allowed temporal gradient (`1.0`: "1.0 K per kilometer")
* `delta_t`: Temperature delta used in the comparison (`2.0`: "2.0 K")
* `n_neighbours`: Number of neighboring points considered in the analysis (`5`)

In [ ]:
qc_spike = do_spike_check(
    value=data.sst,
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    max_gradient_space=0.5,
    max_gradient_time=1.0,
    delta_t=2.0,
    n_neighbours=5,
)
pd.DataFrame({"lat": data.lat, "lon": data.lon, "date": data.date, "sst": data.sst, "qc_spike": qc_spike})

In the plot above, we can see three obvious spikes that are represented in the results of the spike check (`1` (failed)). The spike checks gives the expected results.

Now, we do the track check. As for the spike check we need some extra parameters:

* `speed_limit`: Speed limit of platform in kilometers per hour (`25.0`)
* `delta_d`: Latitude tolerance in degrees (`1.11`)
* `delta_t`: Time tolerance in hundredths of an hour (`0.01`)
* `n_neighbours`: Number of neighbouring points considered in the analysis (`5`)

In [ ]:
qc_track = do_iquam_track_check(
    lat=data.lat,
    lon=data.lon,
    date=data.date,
    speed_limit=25.0,
    delta_d=1.11,
    delta_t=0.01,
    n_neighbours=5,
)
pd.DataFrame({"lat": data.lat, "lon": data.lon, "date": data.date, "qc_track": qc_track})

Within the given parameters, the ship is too "fast" between (44.387443, -28.748580) and (43.730756, -27.658971).

Now, we can run these two QC checks within a single function `do_multiple_sequential_check`. Therefore, we need a `pandas.DataFrame` and a nested QC dictionary. This is the structure of the dictionary:

* arbitrary user-specified name for the checks
  * "func": name of the QC function as `str` (mandatory)
  * "names": dictionary containing parameter names of the QC function and their corresponding columns in the `pandas.DataFrame` (mandatory)
  * "arguments": dictionary containing any key-word arguments passed to the QC function (optionally)
* ...

We define the QC dictionary according to the QC checks above.

In [ ]:
qc_dict = {
    "spike_check": {
        "func": "do_spike_check",
        "names": {
            "value": "sst",
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "max_gradient_space": 0.5,
            "max_gradient_time": 1.0,
            "delta_t": 1.0,
            "n_neighbours": 5,
        },
    },
    "iquam_track_check": {
        "func": "do_iquam_track_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
            "date": "date",
        },
        "arguments": {
            "speed_limit": 25.0,
            "delta_d": 1.11,
            "delta_t": 0.01,
            "n_neighbours": 5,
        },
    },
}

We set `return_method` to "failed" which means that all requested QC check are run until the first check fails. The other QC checks are flagged as unstested (`3`). Furthermore, we set `groupby` to "ship_id" which specifies how the data should be grouped before applying QC functions.

In [ ]:
qc_multi = do_multiple_sequential_check(
    data,
    qc_dict=qc_dict,
    groupby="ship_id",
    return_method="failed",
)
qc_multi

Now, we combine the results into one final QC flag. The QC flag values are prioritized in this order: [1, 0, 3, 2].

In [ ]:
qc_flag = combine_qc_results(qc_multi)
pd.DataFrame({**data, "qc_flag": qc_flag})